# NB2 — Training (config-driven, v4)

Trains LGMF and every ablation from one notebook — edit only the **CONFIG** cell.
v4 adds three levers behind flags: `stem_stride` (resolution), `loss_type`
(dice_bce | focal_tversky), and weight EMA. RNG-restore bug fixed.

**Attach:** `lgmf_c` (code, v4.0.0), Sen1Floods11 imagery, `lgmf-index`.
**Internet: On** (encoder downloads pretrained weights). One run = one `run_name`.


In [ ]:
import sys, os, glob, json, time, random, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "lgmf package not found -- attach the lgmf_c dataset"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.model import build_model, count_params
from lgmf.metrics import flops_g
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__, "| torch", torch.__version__)
assert lgmf.__version__ >= "4.0.0", "attach lgmf_c v4.0.0"

## CONFIG — the only cell you edit per run

**v4 search matrix** (run these to pick the best config, then lock it):
`v4_base` (defaults) → `v4_stride1` → `v4_ft` (focal-tversky) → `v4_stride1_ft`.
After locking the winner, run the 5 ablations with that config.


In [ ]:
cfg = LGMFConfig()

# ===================== EDIT PER RUN =====================
cfg.run_name = "v4_base"
cfg.epochs   = 45
cfg.batch_size = 8
cfg.lr = 3e-4

# ---- v4 architecture/loss search: try these one at a time ----
# cfg.run_name="v4_stride1";     cfg.stem_stride=1                       # finer edges (if OOM: batch_size=4)
# cfg.run_name="v4_ft";          cfg.loss_type="focal_tversky"          # lift SAR-only recall
# cfg.run_name="v4_stride1_ft";  cfg.stem_stride=1; cfg.loss_type="focal_tversky"

# ---- ablations (run AFTER locking the winning v4 config; set that config too) ----
# cfg.run_name="abl_no_film";   cfg.film_on=False
# cfg.run_name="abl_concat";    cfg.gate_mode="concat"
# cfg.run_name="abl_sum";       cfg.gate_mode="sum"
# cfg.run_name="abl_uniform";   cfg.use_curriculum=False
# cfg.run_name="abl_no_cons";   cfg.consistency_weight=0.0
# =======================================================

cfg.ckpt_dir = "/kaggle/working/ckpt"
hits = glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)
assert hits, "norm_stats.json not found -- attach lgmf-index"
cfg.split_dir = os.path.dirname(hits[0])
print("using index dir:", cfg.split_dir)

s = json.load(open(f"{cfg.split_dir}/norm_stats.json"))
cfg.s1_mean, cfg.s1_std = s["s1_mean"], s["s1_std"]
cfg.s2_mean, cfg.s2_std = s["s2_mean"], s["s2_std"]
assert list(s["s2_bands"]) == list(cfg.s2_bands), "S2 band subset mismatch vs NB1!"
print("run:", cfg.run_name, "| split:", s["split_source"])
print(f"flags -> film:{cfg.film_on} gate:{cfg.gate_mode} curric:{cfg.use_curriculum} "
      f"cons:{cfg.consistency_weight} stride:{cfg.stem_stride} loss:{cfg.loss_type} ema:{cfg.use_ema}")

## Data, model, optimizer, EMA

In [ ]:
def seed_all(sd):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
seed_all(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"; print("device:", device)

train_ds = Sen1Floods11(cfg, f"{cfg.split_dir}/train_index.json", augment=True)
val_ds   = Sen1Floods11(cfg, f"{cfg.split_dir}/val_index.json",   augment=False)
train_ld = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=cfg.num_workers, pin_memory=pin, drop_last=True)
val_ld   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                      num_workers=cfg.num_workers, pin_memory=pin)
print(f"train {len(train_ds)} | val {len(val_ds)}")

model = build_model(cfg)
g = flops_g(model, cfg, device="cpu")
print(f"params {count_params(model):.2f}M | GFLOPs@512 {g:.1f}" if g > 0
      else f"params {count_params(model):.2f}M")
model = model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type == "cuda") else None
ema = ModelEMA(model, cfg.ema_decay) if cfg.use_ema else None
eval_model = ema.module if ema else model      # select & report on EMA weights

## Checkpoint & resume (EMA-aware, RNG fixed)

In [ ]:
CKPT = os.path.join(cfg.ckpt_dir, cfg.run_name); os.makedirs(CKPT, exist_ok=True)

def save_ckpt(path, epoch, best, history):
    torch.save({
        "epoch": epoch, "best": best, "history": history,
        "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
        "scaler": scaler.state_dict() if scaler else None,
        "ema": ema.module.state_dict() if ema else None,
        "cfg": dict(cfg.__dict__),
        "rng": {"torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                "numpy": np.random.get_state(), "python": random.getstate()},
    }, path)

def load_ckpt(path):
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    if scaler and ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
    if ema is not None and ck.get("ema"): ema.module.load_state_dict(ck["ema"])
    r = ck["rng"]
    torch.set_rng_state(r["torch"].to("cpu", torch.uint8))          # <-- RNG fix
    if r["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([x.to("cpu", torch.uint8) for x in r["cuda"]])
    np.random.set_state(r["numpy"]); random.setstate(r["python"])
    return ck["epoch"], ck["best"], ck["history"]

HKEYS = ["train_loss", "val_iou_mean", "val_iou_both", "val_iou_s1", "val_iou_s2"]
start_epoch, best, history = 0, -1.0, {k: [] for k in HKEYS}
resume = os.path.join(CKPT, "last.pt")
if not os.path.exists(resume):
    prev = glob.glob(f"/kaggle/input/**/{cfg.run_name}/last.pt", recursive=True)
    resume = prev[0] if prev else resume
if os.path.exists(resume):
    start_epoch, best, history = load_ckpt(resume); start_epoch += 1
    print(f"resumed from {resume} at epoch {start_epoch} (best {best:.4f})")
else:
    print("starting fresh")

## Train

In [ ]:
regs = ["both", "s1_only", "s2_only"]
for epoch in range(start_epoch, cfg.epochs):
    t0 = time.time()
    tr = train_one_epoch(model, train_ld, opt, cfg, epoch, device, scaler=scaler, ema=ema)
    sched.step()
    val = evaluate(eval_model, val_ld, cfg, device)      # eval on EMA weights
    score = float(np.mean([val[r]["iou"] for r in regs]))
    history["train_loss"].append(tr["loss"]); history["val_iou_mean"].append(score)
    history["val_iou_both"].append(val["both"]["iou"])
    history["val_iou_s1"].append(val["s1_only"]["iou"])
    history["val_iou_s2"].append(val["s2_only"]["iou"])
    is_best = score > best
    if is_best: best = score
    save_ckpt(os.path.join(CKPT, "last.pt"), epoch, best, history)
    if is_best: save_ckpt(os.path.join(CKPT, "best.pt"), epoch, best, history)
    print(f"ep {epoch:02d} {time.time()-t0:4.0f}s | loss {tr['loss']:.3f} "
          f"(seg {tr['seg']:.3f} cons {tr['cons']:.4f}) | val IoU "
          f"both {val['both']['iou']:.3f} s1 {val['s1_only']['iou']:.3f} "
          f"s2 {val['s2_only']['iou']:.3f} mean {score:.3f}" + ("  *best" if is_best else ""))
print("done. best mean-IoU:", round(best, 4))

## Evaluate best checkpoint + save results

In [ ]:
load_ckpt(os.path.join(CKPT, "best.pt"))
val = evaluate(eval_model, val_ld, cfg, device)
print("BEST (EMA) — validation:")
for r in regs:
    m = val[r]
    extra = f"  flood-only {m['flood_only_iou']:.3f}" if "flood_only_iou" in m else ""
    print(f"  {r:8s} IoU {m['iou']:.3f}  F1 {m['f1']:.3f}{extra}")
json.dump({"run": cfg.run_name, "best_score": best, "val": val,
           "history": history, "config": dict(cfg.__dict__)},
          open(os.path.join(CKPT, "results.json"), "w"), indent=2, default=str)
cfg.save(os.path.join(CKPT, "config.json"))
print("saved results.json + config.json to", CKPT)

## Training curves

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("/kaggle/working/figs", exist_ok=True)
ep = range(len(history["train_loss"]))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ep, history["train_loss"]); ax[0].set_title("train loss"); ax[0].grid(alpha=0.3)
for k, lab in [("val_iou_both","both"),("val_iou_s1","SAR only"),
               ("val_iou_s2","optical only"),("val_iou_mean","mean")]:
    ax[1].plot(ep, history[k], label=lab)
ax[1].set_title("val IoU by regime"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
p = f"/kaggle/working/figs/{cfg.run_name}_curves"
fig.savefig(p+".png", dpi=200, bbox_inches="tight"); fig.savefig(p+".pdf", bbox_inches="tight")
plt.show(); print("saved", p)

## Next
1. Run the **4 search configs** (`v4_base`, `v4_stride1`, `v4_ft`, `v4_stride1_ft`), ~45 ep each.
2. Pick the winner by best mean-IoU (and check SAR-only specifically).
3. Set that winning config as the default, then run the **5 ablations**.
4. Bundle every `ckpt/<run>/` into `lgmf-ckpts` for **NB4**.
